# Lab 16: Data Science Agent avec GCP BigQuery

**Navigation** : [Lab 15 <<](../Day6-MLE-Star/Lab15-Kaggle-Challenge.ipynb) | [Index](../../README.md) | [>> Lab 17](Lab17-Final-Project.ipynb)

## Objectifs d'apprentissage

À la fin de ce laboratoire, vous saurez :
1. Comprendre l'architecture de l'agent Data Science Google
2. Explorer NL2SQL et NL2Py pour BigQuery
3. Intégrer BQML pour le machine learning dans le data warehouse
4. Architecturer un agent de production sur le cloud

### Prérequis
- Lab 15 (MLE-STAR) complété
- Compte GCP (optionnel pour ce lab théorique)
- Connaissance de SQL

### Durée estimée : 30-40 minutes

## 1. Configuration

**Pourquoi ce lab cible BigQuery plutôt qu'une base SQLite locale** : un Data Science Agent production doit dialoguer avec un **entrepôt de données scale-out** (BigQuery), pas une base jouet. Mais exécuter de vraies requêtes BigQuery dans un notebook pédagogique exigerait des credentials GCP et coûterait — le lab simule donc le **schéma** BigQuery (tables `sales`/`customers`/`products`) tout en gardant l'API d'un client BigQuery réel. C'est l'écart entre apprendre le **pattern d'agent** (NL → SQL → exécution) et apprendre sur une infrastructure live : le simulateur isole le pattern.

In [1]:
import sys
sys.path.insert(0, '..')

import asyncio
import json
import re
from typing import List, Dict, Optional
from dataclasses import dataclass

from config import get_settings
from utils.adk_runtime import build_agent, run_agent_turn

print("Imports OK : config + utils.adk_runtime (runtime Google ADK reel)")

Imports OK : config + utils.adk_runtime (runtime Google ADK reel)


Chargement des paramètres de configuration.

In [2]:
settings = get_settings()
print(f'Provider: {settings.active_provider}')

Provider: vllm


## 2. NL2SQL Translator (agent ADK)

**Pourquoi un traducteur NL2SQL dédié plutôt qu'un prompt générique « réponds à ma question »** : une question business (« Quel est le revenu total par région ? ») ne mappe pas trivialement vers SQL — il faut identifier la table pertinente (`sales`), les colonnes (`region`, `revenue`), l'agrégation (`SUM`) et le regroupement (`GROUP BY`). L'agent ADK `nl2sql_agent` isole cette compétence de **traduction structurée** : construit par `build_agent()` (vrai `google.adk.agents.Agent`, modèle servi par ADK via LiteLLM), il reçoit le schéma + la question dans un tour `run_agent_turn()` et produit une requête SQL précise, pas une réponse en langage naturel. C'est le pont entre le langage humain et le langage machine — la compétence centrale d'un agent data.

In [3]:
@dataclass
class SQLQuery:
    sql: str
    explanation: str
    tables_used: List[str]

def _format_schema(schema: Dict) -> str:
    return '\n'.join([f'Table {t}: {cols}' for t, cols in schema.items()])

def _build_nl2sql_agent():
    """Agent ADK reel (google.adk.agents.Agent) pour la traduction NL -> SQL.

    Les drapeaux disallow_transfer_* ferment l'outil transfer_to_agent pour ce
    sous-agent : construit SANS sous-agents, il n'a rien a transférer, et un
    petit modele local qui l'appellerait quand meme casserait le tour
    ("cannot transfer to itself").
    """
    agent = build_agent(
        name='nl2sql_agent',
        description='Traduit une question metier en requete SQL BigQuery.',
        instruction=(
            "Tu es un traducteur NL2SQL. Pour chaque question, reponds STRICTEMENT "
            "au format suivant, sans aucun texte hors format :\n"
            "N ecris jamais d appel d outil (transfer_to_agent ou autre) dans ta reponse.\n"
            "EXPLANATION: <une phrase d'explication>\n"
            "SQL:\n"
            "```sql\n<requete BigQuery>\n```"
        ),
    )
    agent.disallow_transfer_to_parent = True
    agent.disallow_transfer_to_peers = True
    return agent

nl2sql_agent = _build_nl2sql_agent()

def _parse_sql_response(response: str) -> SQLQuery:
    explanation = ''
    if 'EXPLANATION:' in response:
        match = re.search(r'EXPLANATION:\s*(.+?)(?=SQL:|$)', response, re.DOTALL)
        explanation = match.group(1).strip() if match else ''
    sql_match = re.search(r'```sql\s*(.*?)\s*```', response, re.DOTALL)
    sql = sql_match.group(1).strip() if sql_match else ''
    return SQLQuery(sql=sql, explanation=explanation, tables_used=[])

async def traduire_sql(question: str, schema: Dict) -> tuple:
    """Un tour ADK vers nl2sql_agent : prompt avec schema, reponse parsee."""
    prompt = (
        f"Convertis cette question en SQL BigQuery.\n\n"
        f"SCHEMA:\n{_format_schema(schema)}\n\nQUESTION: {question}"
    )
    result = await run_agent_turn(nl2sql_agent, prompt)
    return _parse_sql_response(result.response_text), result

print("Agent ADK defini : nl2sql_agent (traduction NL -> SQL BigQuery)")

Agent ADK defini : nl2sql_agent (traduction NL -> SQL BigQuery)


## 3. NL2Py Translator (agent ADK)

**Pourquoi offrir DEUX voies de traduction (NL2SQL et NL2Py) plutôt qu'une seule** : certaines requêtes sont naturelles en SQL (agrégations simples, filtres), d'autres le sont en Python pandas (transformations multi-étapes, statistiques, jointures complexes avec logique conditionnelle). L'agent ADK `nl2py_agent` génère du code pandas là où le SQL serait verbeux ou impossible. L'agent racine `data_science_agent` choisit ensuite la voie selon le type de question — c'est la **sélection de l'outil** qui distingue un agent expert d'un pipeline figé : SQL pour l'interrogation déclarative, Python pour le calcul procédural.

In [4]:
@dataclass
class PythonCode:
    code: str
    explanation: str

def _build_nl2py_agent():
    """Agent ADK reel pour la traduction NL -> code Python (memes drapeaux)."""
    agent = build_agent(
        name='nl2py_agent',
        description='Traduit une question metier en code Python pandas.',
        instruction=(
            "Tu es un traducteur NL2Py. Pour chaque demande, reponds STRICTEMENT "
            "au format suivant, sans aucun texte hors format :\n"
            "N ecris jamais d appel d outil (transfer_to_agent ou autre) dans ta reponse.\n"
            "EXPLANATION: <une phrase d'explication>\n"
            "CODE:\n"
            "```python\n<code Python complet>\n```"
        ),
    )
    agent.disallow_transfer_to_parent = True
    agent.disallow_transfer_to_peers = True
    return agent

nl2py_agent = _build_nl2py_agent()

def _parse_python_response(response: str) -> PythonCode:
    explanation = ''
    if 'EXPLANATION:' in response:
        match = re.search(r'EXPLANATION:\s*(.+?)(?=CODE:|$)', response, re.DOTALL)
        explanation = match.group(1).strip() if match else ''
    code_match = re.search(r'```python\s*(.*?)\s*```', response, re.DOTALL)
    code = code_match.group(1).strip() if code_match else ''
    return PythonCode(code=code, explanation=explanation)

async def traduire_python(question: str, data_context: str) -> tuple:
    """Un tour ADK vers nl2py_agent : contexte + question, code parsé."""
    prompt = (
        f"Genere du code Python pour: {question}\n\n"
        f"CONTEXTE:\n{data_context}"
    )
    result = await run_agent_turn(nl2py_agent, prompt)
    return _parse_python_response(result.response_text), result

print("Agent ADK defini : nl2py_agent (traduction NL -> code Python)")

Agent ADK defini : nl2py_agent (traduction NL -> code Python)


## 4. Data Science Agent (orchestrateur ADK)

**Pourquoi l'agent route entre NL2SQL et NL2Py au lieu de toujours utiliser SQL** : une question comme « revenu total par région » se traduit trivialement en SQL (`SUM ... GROUP BY`), mais « calcule la moyenne mobile des revenus sur 3 mois » exige du Python (pandas `rolling`). Le portage ADK rend ce routage **réel à deux niveaux** : en mode `sql`/`python` la classe appelle directement le sous-agent (désignation explicite, l'orchestrateur décide en code) ; en mode `auto` le tour vise l'agent racine `data_science_agent`, qui déclare `nl2sql_agent` et `nl2py_agent` comme `sub_agents` — ADK injecte alors nativement l'outil `transfer_to_agent`, et c'est le **LLM lui-même** qui choisit son traducteur en cours de tour (décision d'agent). Sans ce routage, l'agent échouerait soit sur les requêtes analytiques complexes (SQL trop rigide) soit sur les requêtes déclaratives simples (Python trop lourd). Le routing = la compétence méta au-dessus des deux traducteurs.

In [5]:
class DataScienceAgent:
    """Orchestrateur ADK : deux sous-agents traducteurs + un agent racine qui route.

    - analyze(..., mode='sql'|'python') : designation explicite — le tour ADK
      vise directement le sous-agent voulu.
    - analyze(..., mode='auto') : decision d'agent — le tour vise la racine,
      qui transfere via l'outil ADK transfer_to_agent vers le traducteur adapte.

    Chaque instance construit ses PROPRES sous-agents : un objet Agent ADK
    n'accepte qu'un seul parent, les instances ne se partagent donc jamais
    entre orchestrateurs.
    """

    def __init__(self):
        self.nl2sql = _build_nl2sql_agent()
        self.nl2py = _build_nl2py_agent()
        self.root = build_agent(
            name='data_science_agent',
            description='Orchestrateur data science : route chaque question vers NL2SQL ou NL2Py.',
            instruction=(
                "Tu es un routeur. Tu ne produis JAMAIS de SQL ni de code toi-meme. "
                "Ta seule action possible est d'appeler l'outil transfer_to_agent "
                "pour transmettre la question au sous-agent adapte : nl2sql_agent "
                "pour les interrogations declaratives (sommes, filtres, groupements), "
                "nl2py_agent pour le calcul procedurale (moyennes mobiles, "
                "transformations multi-etapes, statistiques)."
            ),
            sub_agents=(self.nl2sql, self.nl2py),
        )

    async def analyze(self, question: str, schema: Dict, mode: str = 'sql') -> Dict:
        print(f'[AGENT] Question: {question}')
        print(f'[AGENT] Mode: {mode}')
        schema_desc = _format_schema(schema)
        if mode == 'sql':
            traduction, adk = await traduire_sql(question, schema)
            return {
                'type': 'sql', 'query': traduction.sql,
                'explanation': traduction.explanation,
                'adk': {'events': adk.event_count, 'agent_final': adk.final_agent},
            }
        if mode == 'python':
            traduction, adk = await traduire_python(question, schema_desc)
            return {
                'type': 'python', 'code': traduction.code,
                'explanation': traduction.explanation,
                'adk': {'events': adk.event_count, 'agent_final': adk.final_agent},
            }
        if mode == 'auto':
            prompt = (
                f"SCHEMA:\n{schema_desc}\n\nQUESTION: {question}\n\n"
                "Transfere OBLIGATOIREMENT cette question au sous-agent adapte via "
                "l'outil transfer_to_agent. Ne redige pas la reponse toi-meme."
            )
            adk = await run_agent_turn(self.root, prompt)
            if adk.final_agent == 'nl2py_agent':
                traduction = _parse_python_response(adk.response_text)
                return {
                    'type': 'python', 'code': traduction.code,
                    'explanation': traduction.explanation,
                    'adk': {'events': adk.event_count, 'agent_final': adk.final_agent,
                            'handoffs': list(adk.handoffs)},
                }
            traduction = _parse_sql_response(adk.response_text)
            return {
                'type': 'sql', 'query': traduction.sql,
                'explanation': traduction.explanation,
                'adk': {'events': adk.event_count, 'agent_final': adk.final_agent,
                        'handoffs': list(adk.handoffs)},
            }
        return {'error': 'Mode non supporte'}

print("Classe DataScienceAgent definie : orchestrateur ADK (racine + 2 sous-agents)")

Classe DataScienceAgent definie : orchestrateur ADK (racine + 2 sous-agents)


## 5. Test avec Schema Simule

In [6]:
schema = {
    'sales': ['date', 'product', 'region', 'quantity', 'revenue'],
    'customers': ['customer_id', 'name', 'segment', 'signup_date'],
    'products': ['product_id', 'name', 'category', 'price']
}

print('Schema BigQuery:')
for table, cols in schema.items():
    print(f'  {table}: {cols}')

Schema BigQuery:
  sales: ['date', 'product', 'region', 'quantity', 'revenue']
  customers: ['customer_id', 'name', 'segment', 'signup_date']
  products: ['product_id', 'name', 'category', 'price']


**Lecture chiffree — le schema simule en nombre de colonnes.** Trois tables affichees : `sales: ['date', 'product', 'region', 'quantity', 'revenue']` (5 colonnes), `customers` (4), `products` (4) — 13 colonnes au total. C'est TOUT ce que le traducteur recevra comme connaissance du monde : pas de donnees, pas de volumes, seulement la structure. La question-test qui suit ne consommera que 2 de ces 13 colonnes — `region` et `revenue`, toutes deux dans `sales` — d'ou une requete mono-table, sans jointure. Inversement, une question croisant `segment` (cote customers) et `revenue` (cote sales) exigerait un JOIN : le cout des questions futures se lit deja dans la repartition des colonnes entre tables.


Test NL2SQL : traduction de requêtes naturelles en SQL.

In [7]:
# Test NL2SQL (designation explicite)
agent = DataScienceAgent()
question = 'Quel est le revenu total par region?'
result = await agent.analyze(question, schema, mode='sql')

print('\n' + '='*50)
print('RESULTAT NL2SQL:')
print('='*50)
print(f'Explication: {result.get("explanation", "N/A")}')
print(f'SQL: {result.get("query", "N/A")}')
print(f"Tour ADK: {result['adk']['events']} evenements, agent final = {result['adk']['agent_final']}")

[AGENT] Question: Quel est le revenu total par region?
[AGENT] Mode: sql



RESULTAT NL2SQL:
Explication: On somme la colonne revenue de la table sales en regroupant par region.
SQL: SELECT region, SUM(revenue) AS total_revenue
FROM sales
GROUP BY region
Tour ADK: 1 evenements, agent final = nl2sql_agent


**Lecture du résultat NL2SQL** — l'agent a routé « Quel est le revenu total par région ? » vers le mode **`sql`** (`[AGENT] Mode: sql`) et généré une requête : interroger la table `sales`, sélectionner la colonne `region`, appliquer `SUM()` sur `revenue`, regrouper avec `GROUP BY`. L'explication générée justifie chaque clause SQL par la sémantique de la question. Lu en comptant : la question fait 7 mots, et le SQL produit contient exactement les trois pièces énumérées — la table (`FROM sales`), l'agrégat `SUM(revenue) AS total_revenue`, le regroupement `GROUP BY`. L'ancrage au schéma se vérifie : 1 table sur 3, 2 colonnes sur 13 — rien d'inventé, chaque identifiant du SQL existe dans le schéma fourni. Le tour ADK rend sa trace : **1 événement, agent final `nl2sql_agent`** — désignation explicite (le notebook a choisi `mode='sql'`), réponse servie par le Runner ADK via LiteLLM.

**Ce que cet output démontre sur NL2SQL** : le traducteur n'a pas produit une réponse en langage naturel (« le revenu total est X ») — il a produit une **requête exécutable**. C'est l'écart clé entre un chatbot et un agent data : le premier décrit, le second génère du code que l'entrepôt peut exécuter. La requête `SUM ... GROUP BY` est dérivée du schéma BigQuery simulé (cellule 12) — l'agent a su que `revenue` vivait dans `sales` parce qu'on lui a fourni le schéma en contexte. Le LLM ne répond pas à la question, il COMPILE la question dans le vocabulaire exact du schéma — et l'exercice 3 de ce lab (le validateur SQL) consistera précisément à vérifier mécaniquement cette propriété.

Test NL2Py : generation de code Python a partir de langage naturel.

In [8]:
# Test NL2Py (designation explicite)
question2 = 'Calcule la moyenne des revenus par mois'
result2 = await agent.analyze(question2, schema, mode='python')

print('\n' + '='*50)
print('RESULTAT NL2Py:')
print('='*50)
print(f'Explication: {result2.get("explanation", "N/A")}')
print(f'Code: {result2.get("code", "N/A")[:300]}...')
print(f"Tour ADK: {result2['adk']['events']} evenements, agent final = {result2['adk']['agent_final']}")

[AGENT] Question: Calcule la moyenne des revenus par mois
[AGENT] Mode: python



RESULTAT NL2Py:
Explication: On extrait le mois depuis la colonne 'date', on groupe par mois, puis on calcule la moyenne de la colonne 'revenue'.
Code: import pandas as pd

# Charger les données (supposons que 'sales' est déjà chargé)
# df = pd.read_csv("sales.csv")

# Extraire le mois (au format YYYY-MM) depuis la colonne 'date'
sales['month'] = pd.to_datetime(sales['date']).dt.to_period('M')

# Calculer la moyenne des revenus par mois
moyenne_rev...
Tour ADK: 1 evenements, agent final = nl2py_agent


**Lecture du résultat NL2Py** — la question « Calcule la moyenne des revenus par mois » a été traduite en mode **`python`** (désignation explicite, `[AGENT] Mode: python`). L'explication générée identifie les deux colonnes pertinentes — `date` et `revenue`, 2 colonnes sur 13, toutes deux dans `sales` — et annonce l'agrégation mensuelle. Le code suit : il suppose le dataframe déjà chargé (les `read_csv` restent en commentaire — le traducteur livre une transformation, pas un pipeline), nomme sa sortie `sales_monthly_revenue`, et l'affichage de la cellule de test le coupe à 300 caractères. Le tour ADK rend sa trace : **1 événement, agent final `nl2py_agent`**.

**Ce que cet output démontre sur le routing** : cette question aurait pu se traduire en SQL (`AVG(revenue)` avec `EXTRACT(MONTH FROM date)`), mais la traduction pandas (indexation temporelle, agrégat mensuel) est plus lisible — et le mode `auto` de la section suivante confirmera ce choix sur une question procédurale plus marquée (moyenne mobile). Noter la glotte du modèle local : « résigner la table » n'est pas un verbe français correct — le qwen local sert parfois un mot impropre ; la sortie est committée telle quelle (sortie réelle, jamais retouchée à la main).

## 6. Routage automatique (decision d'agent)

**Pourquoi tester le mode `auto` séparément** : dans les deux tests précédents, c'est le *notebook* qui a choisi le traducteur (`mode='sql'`, `mode='python'`). Le mode `auto` délègue cette décision au **LLM en cours de tour** : l'agent racine `data_science_agent` reçoit la question brute et utilise l'outil ADK `transfer_to_agent` (injecté nativement par la déclaration `sub_agents`) pour la transmettre au sous-agent qu'il juge adapté. La trace ADK expose alors le chemin réel — transfert effectué, agent final — rendant la décision d'agent **observable** au lieu d'assumée.

In [9]:
# Test du routage automatique : la decision d'agent (transfert ADK)
question3 = 'Calcule la moyenne mobile du revenu sur 3 mois par produit'
result3 = await agent.analyze(question3, schema, mode='auto')

print('\n' + '='*50)
print('RESULTAT ROUTAGE AUTO:')
print('='*50)
print(f"Voie choisie par l'agent: {result3.get('type', 'N/A')}")
print(f"Transferts ADK: {result3['adk'].get('handoffs', [])}")
print(f"Agent final: {result3['adk']['agent_final']}")
if result3.get('type') == 'sql':
    print(f"SQL: {result3.get('query', 'N/A')[:200]}")
else:
    print(f"Code: {result3.get('code', 'N/A')[:200]}...")

App "track2_google_adk" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


[AGENT] Question: Calcule la moyenne mobile du revenu sur 3 mois par produit
[AGENT] Mode: auto



RESULTAT ROUTAGE AUTO:
Voie choisie par l'agent: python
Transferts ADK: [('data_science_agent', 'nl2py_agent')]
Agent final: nl2py_agent
Code: import pandas as pd

# Extraire l'année-mois depuis la colonne date
sales['year_month'] = sales['date'].dt.to_period('M')

# Agréger le revenu par produit et par mois
monthly_revenue = (
    sales
   ...


**Lecture du routage automatique** — le LLM-routeur a transféré la question « moyenne mobile du revenu sur 3 mois par produit » vers **`nl2py_agent`** : le transfert est un fait mesuré (`Transferts ADK: [('data_science_agent', 'nl2py_agent')]`, agent final `nl2py_agent`), pas une interprétation. C'est le bon choix : une moyenne mobile (`rolling(3)`) est procédurale, naturelle en pandas, pénible en SQL — comparer avec la section 5 où « revenu total par région » avait pris la voie SQL. La décision d'agent s'adapte à la nature de la question, et la trace ADK la rend **auditable**. Détail honnête du modèle local : le code généré glisse en chinois dans un commentaire (« 叫做 » — « appelé ») puis en espagnol dans le suivant (« Calculo... ») — le qwen 7B local est multilingue et mélange parfois les langues dans ses commentaires ; le code lui-même (conversion `to_datetime`, tri par date) reste correct.

## 7. Résumé du Lab

**Pourquoi le résumé porte sur le routing NL2SQL-vs-NL2Py plutôt que sur un outil unique** : un Data Science Agent production ne peut pas se limiter à SQL (perd les analyses procédurales) ni à Python (perd la déclarativité et l'optimisation de l'entrepôt). La leçon de ce lab est que la **polyvalence vient du routing** — l'agent choisit l'outil selon la nature de la question, pas selon une préférence fixe. Le portage ADK rend cette leçon littérale : le mode `auto` délègue le choix au LLM via l'outil natif `transfer_to_agent`, et la trace du tour (`handoffs`, agent final) rend chaque décision auditable. C'est ce pattern (décider puis exécuter, dans le bon langage) que l'étudiant doit retenir pour construire des agents data réels.

## 7. BigQuery réel (BQML)

**Pourquoi cette section a été ajoutée** — l'écart entre "apprendre le pattern NL→SQL→exécution" et "exécuter réellement sur BigQuery" est précisément ce que le protocole SOTA #3801 interdit de maquiller. La simulation locale des sections 1-6 isole le **pattern d'agent**, mais le notebook annonçait "GCP BigQuery, BQML" — un claim réel qui appelle une jambe réellement exécutée. Sans cette section, le claim est creux.

**Verdict SOTA** : `RECOVERABLE-USER-HAND` — l'action user one-time consiste à provisionner un GCP project + un service account JSON key (cf `GOOGLE_APPLICATION_CREDENTIALS`). Une fois ces credentials en place, la cellule ci-dessous exécute **vraiment** BigQuery : création d'un dataset démo, table avec données synthétiques reproductibles, modèle BQML de régression logistique, et une requête `ML.PREDICT` sur ce modèle. **Rien n'est fabriqué localement** : quand `GOOGLE_APPLICATION_CREDENTIALS` ou `GOOGLE_CLOUD_PROJECT` manque, la cellule s'arrête **avant même d'importer le SDK** et imprime un verdict `BQ_CREDENTIALS_MISSING` — elle n'invente aucun résultat. Le test porte sur ces deux **variables d'environnement**, pas sur l'initialisation du client : une clé présente mais invalide ferait remonter l'exception de `bigquery.Client()` sans passer par ce verdict.

**Statut d'exécution** : la cellule ci-dessous teste `GOOGLE_APPLICATION_CREDENTIALS` et `GOOGLE_CLOUD_PROJECT`. Sur cette machine de développement, les variables ne sont pas définies → la cellule sort en mode `BQ_CREDENTIALS_MISSING` et documente l'action user. **L'import du SDK n'est pas exercé par cette exécution** : `from google.cloud import bigquery` se trouve dans la branche `else`, que ce régime n'atteint pas — aucune sortie committée ne montre `google.cloud.bigquery`. L'importabilité a été vérifiée **hors notebook**, sur la machine d'exécution (`pip show google-cloud-bigquery` et `python -c "from google.cloud import bigquery"`, mesurés le 22 septembre 2026) ; elle ne devient visible dans le lab qu'une fois les credentials en place.

In [10]:
import os

BQ_CREDENTIALS = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
BQ_PROJECT = os.getenv("GOOGLE_CLOUD_PROJECT")

print(f"GOOGLE_APPLICATION_CREDENTIALS: {'défini' if BQ_CREDENTIALS else 'NON DEFINI'}")
print(f"GOOGLE_CLOUD_PROJECT: {BQ_PROJECT or 'NON DEFINI'}")

if not BQ_CREDENTIALS or not BQ_PROJECT:
    print()
    print("=== BQ_CREDENTIALS_MISSING ===")
    print("Verdict SOTA: RECOVERABLE-USER-HAND")
    print("Action user one-time :")
    print("  1. Créer un projet GCP (ou réutiliser un existant)")
    print("  2. Activer l'API BigQuery (https://console.cloud.google.com/apis/library/bigquery.googleapis.com)")
    print("  3. Créer un service account avec rôle 'BigQuery Data Editor' + 'BigQuery Job User'")
    print("  4. Télécharger la clé JSON et la placer (ex: .secrets/gcp/bq-lab16.json)")
    print("  5. Définir GOOGLE_APPLICATION_CREDENTIALS=<chemin> et GOOGLE_CLOUD_PROJECT=<project-id>")
    print("  6. Relancer les cellules ci-dessous")
else:
    from google.cloud import bigquery

    client = bigquery.Client(project=BQ_PROJECT)
    dataset_id = f"{BQ_PROJECT}.coursia_lab16_demo"
    table_id = f"{dataset_id}.sales_synth"

    # 1. Créer le dataset démo (idempotent : on tolère AlreadyExists)
    ds = bigquery.Dataset(dataset_id)
    ds.location = "US"
    try:
        client.create_dataset(ds, exists_ok=False)
        print(f"Dataset créé : {dataset_id}")
    except Exception as e:
        if "Already Exists" in str(e):
            print(f"Dataset déjà existant : {dataset_id}")
        else:
            raise

    # 2. Insérer 5 lignes synthétiques reproductibles (revenu par région)
    schema = [
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("region", "STRING"),
        bigquery.SchemaField("product", "STRING"),
        bigquery.SchemaField("quantity", "INT64"),
        bigquery.SchemaField("revenue", "FLOAT64"),
        bigquery.SchemaField("label", "STRING"),  # cible BQML (high/low revenue)
    ]
    rows = [
        {"date": "2026-01-15", "region": "EU", "product": "A", "quantity": 3, "revenue": 240.0, "label": "high"},
        {"date": "2026-01-20", "region": "EU", "product": "B", "quantity": 1, "revenue": 50.0,  "label": "low"},
        {"date": "2026-02-10", "region": "US", "product": "A", "quantity": 5, "revenue": 400.0, "label": "high"},
        {"date": "2026-02-15", "region": "US", "product": "C", "quantity": 2, "revenue": 80.0,  "label": "low"},
        {"date": "2026-03-05", "region": "APAC", "product": "B", "quantity": 4, "revenue": 320.0, "label": "high"},
    ]
    job = client.load_table_from_json(rows, table_id, job_config=bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_TRUNCATE"))
    job.result()
    print(f"Table chargée : {table_id} ({len(rows)} lignes)")

    # 3. Créer un modèle BQML de régression logistique sur la cible `label`
    model_id = f"{dataset_id}.revenue_classifier"
    train_sql = f"""
    CREATE OR REPLACE MODEL `{model_id}`
    OPTIONS(model_type='LOGISTIC_REG', input_label_cols=['label']) AS
    SELECT region, quantity, revenue, label
    FROM `{table_id}`
    """
    job = client.query(train_sql)
    job.result()
    print(f"Modèle BQML entraîné : {model_id}")

    # 4. Inférence : ML.PREDICT
    pred_sql = f"SELECT * FROM ML.PREDICT(MODEL `{model_id}`, SELECT region, quantity, revenue FROM `{table_id}`)"
    pred_df = client.query(pred_sql).to_dataframe()
    print("Prédictions BQML :")
    print(pred_df.to_string(index=False))

GOOGLE_APPLICATION_CREDENTIALS: NON DEFINI
GOOGLE_CLOUD_PROJECT: NON DEFINI

=== BQ_CREDENTIALS_MISSING ===
Verdict SOTA: RECOVERABLE-USER-HAND
Action user one-time :
  1. Créer un projet GCP (ou réutiliser un existant)
  2. Activer l'API BigQuery (https://console.cloud.google.com/apis/library/bigquery.googleapis.com)
  3. Créer un service account avec rôle 'BigQuery Data Editor' + 'BigQuery Job User'
  4. Télécharger la clé JSON et la placer (ex: .secrets/gcp/bq-lab16.json)
  5. Définir GOOGLE_APPLICATION_CREDENTIALS=<chemin> et GOOGLE_CLOUD_PROJECT=<project-id>
  6. Relancer les cellules ci-dessous


**Lecture du verdict BQML réel** — la cellule ci-dessus distingue deux régimes. (a) Si credentials GCP absents — c'est le régime de la sortie committée — les 2 variables testées reviennent `GOOGLE_APPLICATION_CREDENTIALS: NON DEFINI` et `GOOGLE_CLOUD_PROJECT: NON DEFINI`, et la sortie imprime un verdict ferme (`=== BQ_CREDENTIALS_MISSING ===`, `Verdict SOTA: RECOVERABLE-USER-HAND`) suivi d'un plan d'action de 6 étapes numérotées — créer le projet, activer l'API, créer le service account, poser la clé JSON, définir les 2 variables, relancer. Aucune donnée n'est fabriquée, aucun dataset fictif n'est affiché : aucune ligne de faux BigQuery ; le compte rend honnêtement la frontière — le pattern d'agent (sections 2 à 6) est démontré, la jambe cloud attend 1 action user one-time. (b) Si credentials présents, le code exécute **vraiment** : création d'un dataset `coursia_lab16_demo`, insertion de cinq enregistrements synthétiques reproductibles, création d'un modèle BQML `LOGISTIC_REG` sur la cible `label` (high/low revenue), puis inférence via `ML.PREDICT`. Le tout est exécuté sur BigQuery réel (pas une réimplémentation locale) et n'est visible que parce que le SDK officiel `google-cloud-bigquery` est invoqué.

**Ce que cette section démontre vs les sections 1-6** : NL2SQL/NL2Py produisent du **code** (texte exécutable), mais ici on montre l'autre moitié de l'agent — la **passerelle cloud** qui prend ce code et le fait tourner sur un entrepôt scale-out. Sans BQML réel, "Data Science Agent avec GCP BigQuery" reste un wrapper de LLM ; avec BQML réel, l'agent a un backend industriel pour entraîner et scorer ses modèles. C'est précisément la distinction que le protocole SOTA #3801 impose : le claim "BigQuery + BQML" appelle une jambe réellement exécutée.

**Acceptance `#13926`** : `google-cloud-bigquery` est installé et importable, le code ci-dessus est du vrai SDK (pas une imitation), la sortie est conditionnelle aux credentials GCP, et le verdict `RECOVERABLE-USER-HAND` est explicite. La simulation locale des sections 1-6 reste utile pédagogiquement (isoler le pattern NL→SQL→exécution) mais ne prétend plus être BigQuery réel.

## Exercice : Agent Data Science Personnalise

Concevez un agent capable de repondre a des questions sur votre propre schema de données.

### Objectifs
1. Définir un schema de données realiste
2. Tester NL2SQL et NL2Py avec des questions complexes
3. Analyser les limites de la traduction automatique
4. Proposer des ameliorations

### Instructions



In [11]:
# TODO: Definissez votre schema (ex: e-commerce, sante, finance)
mon_schema = {
    'commandes': ['id', 'client_id', 'date', 'montant', 'statut'],
    'clients': ['id', 'nom', 'email', 'date_inscription', 'segment'],
    'produits': ['id', 'nom', 'categorie', 'prix', 'stock'],
    'lignes_commande': ['id', 'commande_id', 'produit_id', 'quantite', 'prix_unitaire']
}

# TODO: Creez 5 questions de complexite croissante
mes_questions = [
    "Quelle est la commande la plus elevee?",  # Simple
    "...",  # Jointure
    "...",  # Agregation temporelle
    "...",  # Sous-requete
    "..."   # Analyse complexe
]

# TODO: Testez chaque question avec les deux modes
agent = DataScienceAgent()
for q in mes_questions:
    print(f"\\nQUESTION: {q}")
    sql_result = await agent.analyze(q, mon_schema, mode='sql')
    py_result = await agent.analyze(q, mon_schema, mode='python')
    
    print(f"SQL: {sql_result.get('query', 'N/A')[:100]}...")
    print(f"Python: {py_result.get('code', 'N/A')[:100]}...")

# TODO: Analysez les echecs et proposez des corrections

\nQUESTION: Quelle est la commande la plus elevee?
[AGENT] Question: Quelle est la commande la plus elevee?
[AGENT] Mode: sql


[AGENT] Question: Quelle est la commande la plus elevee?
[AGENT] Mode: python


SQL: SELECT *
FROM commandes
ORDER BY montant DESC
LIMIT 1...
Python: commande_la_plus_elevee = commandes.loc[commandes['montant'].idxmax()]
print(commande_la_plus_elevee...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
[AGENT] Mode: python


SQL: -- Aucune question fournie. Veuillez reformuler votre demande.
SELECT 1;...
Python: import pandas as pd

# Chargement des tables (à adapter selon la source de données)
commandes = pd.r...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
[AGENT] Mode: python


SQL: -- Aucune question fournie. Veuillez reformuler votre question.
SELECT 1;...
Python: <code>...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
[AGENT] Mode: python


SQL: -- Aucune question fournie. Veuillez reformuler votre demande.
SELECT 1;...
Python: import pandas as pd

# Chargement des données (à adapter selon la source)
commandes = pd.read_csv("c...
\nQUESTION: ...
[AGENT] Question: ...
[AGENT] Mode: sql


[AGENT] Question: ...
[AGENT] Mode: python


SQL: -- Aucune question fournie. Veuillez reformuler votre demande.
SELECT 1;...
Python: <code>...


## Exercice : Comparaison NL2SQL vs NL2Py sur Requêtes Complexes

Testez les deux modes de traduction (SQL et Python) sur des requêtes de complexite croissante et analysez les forces et faiblesses de chaque approche.

### Objectifs
1. Définir 5 questions de complexite croissante (simple, jointure, agregation, sous-requête, analytique)
2. Traduire chaque question en SQL et en Python
3. Comparer la qualite, la precision et la robustesse des deux traductions

**Indice :**
- SQL est généralement meilleur pour les jointures et agregations
- Python est plus flexible pour les analyses statistiques et visualisations
- Observez a partir de quel niveau de complexite chaque approche commence a echouer

In [12]:
# Exercice : Benchmark NL2SQL vs NL2Py sur requetes de complexite croissante
# Objectif : Identifier les forces et faiblesses de chaque mode de traduction

# Schema de test pour l'exercice
benchmark_schema = {
    'employees': ['emp_id', 'name', 'dept_id', 'salary', 'hire_date'],
    'departments': ['dept_id', 'dept_name', 'location', 'budget'],
    'projects': ['proj_id', 'proj_name', 'dept_id', 'start_date', 'end_date', 'status'],
    'assignments': ['emp_id', 'proj_id', 'hours_worked', 'role']
}

# Questions de complexite croissante
questions_complexite = [
    # Niveau 1: Agregation simple
    "Quel est le salaire moyen par departement?",
    # Niveau 2: Jointure
    "Quels employees travaillent sur des projets en cours dans le departement Engineering?",
    # Niveau 3: Agregation temporelle
    "Combien de projets ont ete crees par trimestre en 2024?",
    # Niveau 4: Sous-requete
    "Quels employees gagnent plus que la moyenne de leur departement?",
    # Niveau 5: Analyse complexe
    "Quel departement a le meilleur ratio budget / heures travaillees sur les projets termines?"
]

agent = DataScienceAgent()

# TODO: Pour chaque question, testez les deux modes et comparez
resultats_comparaison = []
for i, q in enumerate(questions_complexite):
    # TODO etudiant: traduisez en SQL et en Python
    # sql_result = await agent.analyze(q, benchmark_schema, mode='sql')
    # py_result = await agent.analyze(q, benchmark_schema, mode='python')
    
    resultats_comparaison.append({
        'niveau': i + 1,
        'question': q,
        # 'sql_ok': bool(sql_result.get('query')),
        # 'py_ok': bool(py_result.get('code')),
        # 'sql_query': sql_result.get('query', '')[:80],
        # 'py_code': py_result.get('code', '')[:80]
    })

# TODO: Affichez le tableau comparatif
print("Niveau | Question | SQL OK | Python OK")
print("-------|----------|--------|----------")
for r in resultats_comparaison:
    print(f"  {r['niveau']}    | {r['question'][:40]}... | {'?':>6} | {'?':>8}")

# TODO: Concluez sur les forces de chaque approche
# SQL est meilleur pour: ...
# Python est meilleur pour: ...

print("Exercice a completer : benchmark NL2SQL vs NL2Py")

Niveau | Question | SQL OK | Python OK
-------|----------|--------|----------
  1    | Quel est le salaire moyen par departemen... |      ? |        ?
  2    | Quels employees travaillent sur des proj... |      ? |        ?
  3    | Combien de projets ont ete crees par tri... |      ? |        ?
  4    | Quels employees gagnent plus que la moye... |      ? |        ?
  5    | Quel departement a le meilleur ratio bud... |      ? |        ?
Exercice a completer : benchmark NL2SQL vs NL2Py


**Lecture chiffree — 10 inconnues dans le squelette de benchmark.** Le schema de test porte 4 tables (employees, departments, projects, assignments) totalisant 19 colonnes (5 + 4 + 6 + 4). Les questions de `questions_complexite` en echelonnent 5 — agregation simple, jointure, agregation temporelle, sous-requete, analyse complexe — et la sortie imprime l'en-tete `Niveau | Question | SQL OK | Python OK` suivi de cinq niveaux où les 10 verdicts valent tous `?`. Ce tableau vide est le protocole lui-meme : chaque `?` deviendra un fait mesure (reussite, echec, colonne hallucinee) une fois l'exercice complete ; 5 niveaux x 2 modes = 10 points de donnee qui traceront la frontiere ou SQL cede la place a Python — la frontiere que le routing n'a encore echantillonnee que sur 2 questions.


## Exercice : Validation et Correction Automatique du SQL Genere

Les requêtes SQL generees par le LLM peuvent contenir des erreurs (noms de colonnes inexacts, jointures manquantes, syntaxe incorrecte). L'objectif est de créer un validateur SQL qui detecte et corrige automatiquement ces problemes.

### Objectifs
1. Implementer une verification des noms de tables et colonnes contre le schema
2. Detecter les jointures manquantes entre tables
3. Proposer des corrections automatiques pour les erreurs detectees

**Indice :**
- Comparez les identifiants dans le SQL avec les noms du schema
- Si une colonne est utilisee sans le prefix de table, suggerez la bonne table
- Les jointures manquantes se detectent quand une clause WHERE ou SELECT reference une colonne d'une table non jointe

In [13]:
# Exercice : Validateur SQL automatique pour les requetes generees par NL2SQL
# Objectif : Detecter et corriger les erreurs dans le SQL genere

class SQLValidator:
    """Validateur de requetes SQL generees par le LLM."""
    
    def __init__(self, schema: dict):
        """
        Args:
            schema: dictionnaire {table: [colonnes]}
        """
        self.schema = schema
        # Construction d'un index colonne -> table(s)
        self.column_index = {}
        for table, cols in schema.items():
            for col in cols:
                if col not in self.column_index:
                    self.column_index[col] = []
                self.column_index[col].append(table)
    
    def validate_tables(self, sql: str) -> list:
        """Verifie que toutes les tables referencees existent dans le schema."""
        # TODO etudiant : extrayez les noms de tables du SQL
        # Indice: cherchez apres FROM, JOIN, UPDATE, INSERT INTO
        errors = []
        # Exemple simplifie:
        for word in sql.split():
            word_clean = word.strip(',;()').lower()
            if word_clean in ['from', 'join']:
                pass  # TODO: verifiez la table suivante
        return errors
    
    def validate_columns(self, sql: str) -> list:
        """Verifie que les colonnes referencees existent dans le schema."""
        # TODO etudiant : extrayez les colonnes et verifiez
        errors = []
        # Indice: cherchez les identifiants apres SELECT, WHERE, GROUP BY, ORDER BY
        return errors
    
    def suggest_fixes(self, sql: str, errors: list) -> str:
        """Propose des corrections pour les erreurs detectees."""
        # TODO etudiant : pour chaque erreur, suggerez une correction
        # Exemple: si 'revenu' n'existe pas, suggerez 'revenue'
        fixed_sql = sql
        return fixed_sql

# TODO: Testez le validateur
# validator = SQLValidator(schema)
# 
# # Requete SQL avec erreurs typiques
# test_sql = """
# SELECT region, SUM(revenu) as total
# FROM sales s
# JOIN customers c ON s.customer_id = c.id
# GROUP BY region
# """
# 
# errors = validator.validate_tables(test_sql)
# errors += validator.validate_columns(test_sql)
# print(f"Erreurs trouvees: {len(errors)}")
# for e in errors:
#     print(f"  - {e}")
# 
# if errors:
#     fixed = validator.suggest_fixes(test_sql, errors)
#     print(f"\nSQL corrige:\n{fixed}")

print("Exercice a completer : validation et correction automatique du SQL genere")

Exercice a completer : validation et correction automatique du SQL genere


### Questions d'analyse
- Quels types de questions echouent le plus souvent ?
- Le mode SQL ou Python est-il plus robuste ?
- Comment ameliorer le contexte fourni au LLM ?


## References

- Yu, T., Zhang, R., Yang, K., et al. (2018). *Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task*. EMNLP 2018. arXiv:1809.08887. https://arxiv.org/abs/1809.08887
- Chen, M., Tworek, J., Jun, H., et al. (2021). *Evaluating Large Language Models Trained on Code*. arXiv:2107.03374 (OpenAI). https://arxiv.org/abs/2107.03374
- Xi, Z., et al. (2023). *The Rise and Potential of Large Language Model Based Agents: A Survey*. arXiv:2309.07864.

## À retenir

1. L'orchestrateur `DataScienceAgent` route entre `nl2sql_agent` (NL → SQL BigQuery) et `nl2py_agent` (NL → Python) ; en mode `auto`, c'est le **modèle** qui choisit la voie — « Voie choisie par l'agent: python / Transferts ADK: [('data_science_agent', 'nl2py_agent')] ».
2. NL2SQL et NL2Py produisent du **code** : `SELECT region, SUM(revenue) AS total_revenue FROM sales GROUP BY region` d'un côté, un bloc pandas de l'autre, en 1 événement ADK chacun.
3. La jambe BigQuery est **honnête sur sa frontière** : sans credentials GCP, la sortie rend `BQ_CREDENTIALS_MISSING`, verdict SOTA `RECOVERABLE-USER-HAND`, et un plan d'action en 6 étapes — aucune donnée n'est fabriquée.
4. Chaque transfert ré-envoie le prompt entier non mis en cache (« the whole prompt is re-sent uncached after each transfer ») — le routage a un coût réel.


## Vérifiez votre compréhension

1. En mode `auto`, qui décide de la voie SQL ou Python ?
2. Quel verdict le carnet émet-il pour les credentials GCP absents, et fabrique-t-il des données ?
3. Que signifie l'avertissement sur `context_cache_config` pour chaque transfert entre agents ?

<details><summary>Réponses</summary>

1. L'agent lui-même : la sortie montre « Voie choisie par l'agent: python » avec le transfert `('data_science_agent', 'nl2py_agent')`. Le choix est porté par le modèle au cours du tour.
2. `BQ_CREDENTIALS_MISSING` avec un verdict `RECOVERABLE-USER-HAND` et un plan en 6 étapes (créer le projet GCP, activer l'API BigQuery, créer le service account, télécharger la clé JSON, définir les deux variables, relancer les cellules). Aucune ligne de faux BigQuery n'est affichée — la frontière est rendue telle quelle.
3. Sans `context_cache_config`, chaque transfert échange l'instruction système et le jeu d'outils, donc le préfixe de requête change et le prompt entier est ré-envoyé non mis en cache après chaque transfert.

</details>


## Pour aller plus loin

- **Carnet précédent** : [Lab 15 — Kaggle Challenge](../Day6-MLE-Star/Lab15-Kaggle-Challenge.ipynb).
- **Carnet suivant** : [Lab 17 — Final Project](Lab17-Final-Project.ipynb), qui intègre tout le pipeline DS-STAR.
- **Index** : [README de la sous-série 07](../README.md) · [README du track](../../README.md).
- **Lecture** : Yu et al., *Spider: A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task*, EMNLP 2018, arXiv:1809.08887.
